In [0]:
# ============================================================
# Silver — Source 15: MQTT IoT Sensors
#
# Transformations:
#   - Cast ts ISO string to timestamp
#   - Validate warehouse_id is one of 3 valid warehouses
#   - Validate reading is not null
#   - Normalise sensor_type, unit
#   - Flag anomalies for downstream alerting
#   - Reject null event_id or warehouse_id → quarantine
#   - Deduplicate on event_id
#
# Source:  bronze.src_15_iot.telemetry
# Target:  silver.src_15_iot.telemetry
# Quarantine: silver.quarantine.src_15_iot
# ============================================================

from pyspark.sql import functions as F
from pyspark.sql.types import *
from delta.tables import DeltaTable

BRONZE_CATALOG = 'bronze'
SILVER_CATALOG = 'silver'
TARGET_TABLE = f'{SILVER_CATALOG}.src_15_iot.telemetry'
QUARANTINE_TABLE = f'{SILVER_CATALOG}.quarantine.src_15_iot'

VALID_WAREHOUSES = ['WH-LONDON-01', 'WH-MANC-01', 'WH-BRUM-01']
VALID_SENSOR_TYPES = ['temperature', 'humidity', 'weight', 'motion', 'door', 'co2']

spark.sql(f'CREATE SCHEMA IF NOT EXISTS {SILVER_CATALOG}.src_15_iot')
print('Silver Source 15 IoT Sensors — starting...')


In [0]:
bronze = spark.table(f'{BRONZE_CATALOG}.src_15_iot.telemetry')
total = bronze.count()
print(f'Bronze rows: {total}')

# Cast timestamp
df = bronze.withColumn('ts', F.to_timestamp(F.col('ts')))

# Normalise
df = df \
    .withColumn('sensor_type',  F.lower(F.trim(F.col('sensor_type')))) \
    .withColumn('unit',         F.lower(F.trim(F.col('unit')))) \
    .withColumn('warehouse_id', F.upper(F.trim(F.col('warehouse_id'))))

# Bad rows
bad = df.filter(
    F.col('event_id').isNull() |
    F.col('warehouse_id').isNull() |
    F.col('reading').isNull() |
    F.col('ts').isNull() |
    ~F.col('warehouse_id').isin(VALID_WAREHOUSES)
).withColumn('quarantine_reason', F.lit('failed_validation')) \
 .withColumn('source_table', F.lit('telemetry'))

# Good rows
good = df.filter(
    F.col('event_id').isNotNull() &
    F.col('warehouse_id').isNotNull() &
    F.col('reading').isNotNull() &
    F.col('ts').isNotNull() &
    F.col('warehouse_id').isin(VALID_WAREHOUSES)
).dropDuplicates(['event_id'])

bad_count = bad.count()
good_count = good.count()
anomaly_count = good.filter(F.col('is_anomaly') == True).count()
print(f'IoT telemetry: {total} total → {good_count} clean, {bad_count} quarantined ({bad_count/total*100:.1f}%)')
print(f'Anomalies detected: {anomaly_count}/{good_count} ({anomaly_count/good_count*100:.1f}%)')

# Write
if spark.catalog.tableExists(TARGET_TABLE):
    dt = DeltaTable.forName(spark, TARGET_TABLE)
    dt.alias('t').merge(good.alias('s'), 't.event_id = s.event_id') \
        .whenMatchedUpdateAll().whenNotMatchedInsertAll().execute()
else:
    good.write.format('delta').mode('overwrite').saveAsTable(TARGET_TABLE)
print('✅ Written')

# Quarantine
if bad_count > 0:
    bad.select(
        F.lit('src_15_iot').alias('source'),
        F.col('source_table'),
        F.col('quarantine_reason'),
        F.current_timestamp().alias('quarantined_at'),
        F.to_json(F.struct(*[c for c in bad.columns if c not in ['quarantine_reason','source_table']])).alias('raw_record')
    ).write.format('delta').mode('append').option('mergeSchema','true').saveAsTable(QUARANTINE_TABLE)
    print(f'✅ {bad_count} quarantined')


In [0]:
count = spark.sql(f'SELECT COUNT(*) as cnt FROM {TARGET_TABLE}').collect()[0]['cnt']
print(f'silver.src_15_iot.telemetry: {count} rows')
spark.sql(f"""
    SELECT warehouse_id, sensor_type, COUNT(*) as readings,
           ROUND(AVG(reading),2) as avg_reading,
           SUM(CAST(is_anomaly as INT)) as anomalies
    FROM {TARGET_TABLE}
    GROUP BY warehouse_id, sensor_type
    ORDER BY warehouse_id, sensor_type
""").show()
